# An agent that reads a report and checks a bearing capacity [Exercise]

Exercise: [![Open in Colab](https://img.shields.io/badge/Open%20in-Colab-F9AB00?style=flat-square&logo=googlecolab)](https://colab.research.google.com/github/kks32-courses/ai-geotech/blob/main/docs/07-llm/07c-llm-report-agent-exercise.ipynb) Solution: [![Open in Colab](https://img.shields.io/badge/Open%20in-Colab-F9AB00?style=flat-square&logo=googlecolab)](https://colab.research.google.com/github/kks32-courses/ai-geotech/blob/main/docs/07-llm/07c-llm-report-agent.ipynb)

Retrieval in 7b put the report in front of the model. The model still did any arithmetic itself.
Here the model gets two tools, one that searches the reports and one that computes bearing
capacity, and a loop that runs whichever tool the model asks for and hands back the result. The
model decides what to look up and what to compute. Python does the computing.

Run this cell first. It installs the packages, reads your API key, and names the model.
Locally the key comes from a `.env` file in this folder. On Colab it comes from
Settings > Secrets, or from a prompt.

In [ ]:
!pip install -q openai pypdf chromadb tiktoken python-dotenv
import os
from pathlib import Path
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv()                                   # local: reads .env in this folder if present
if not os.getenv("OPENAI_API_KEY"):
    try:
        from google.colab import userdata       # Colab: Settings > Secrets > OPENAI_API_KEY
        os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
    except Exception:
        from getpass import getpass
        os.environ["OPENAI_API_KEY"] = getpass("OpenAI API key: ")
client = OpenAI()
MODEL = "gpt-5.6-luna"   # $0.20 in / $1.20 out per 1M tokens, developers.openai.com/api/docs/pricing, 2026-09-10

The four site investigation reports live in `docs/` in the repository. On Colab there is no
repository, so this cell downloads `docs.zip` from GitHub and unpacks it into `reports/`.

In [ ]:
REPORTS = Path("docs") if Path("docs").exists() else Path("reports")   # repo checkout vs Colab
if not REPORTS.exists():
    !wget -nc https://raw.githubusercontent.com/kks32-courses/ai-geotech/main/docs/07-llm/docs.zip
    !unzip -n -q docs.zip -d reports
pdf_paths = sorted(REPORTS.glob("*.pdf"))
print([p.name for p in pdf_paths])

This cell is the index from 7b: read the pages, cut them into chunks, embed them, and define
`search_reports`. The Chroma collection sits on disk, so if you already ran 7b in this folder the
embedding step is skipped.

In [ ]:
from pypdf import PdfReader
import chromadb
import tiktoken

pages = []
for path in pdf_paths:
    for n, page in enumerate(PdfReader(str(path)).pages, start=1):
        pages.append({"doc": path.name, "page": n, "text": page.extract_text() or ""})

ENCODER = tiktoken.get_encoding("cl100k_base")


def chunk(pages, size=400, overlap=50):
    """Cut each page into windows of `size` tokens that overlap by `overlap` tokens."""
    out = []
    for p in pages:
        tokens = ENCODER.encode(p["text"])
        start = 0
        while start < len(tokens):
            end = min(len(tokens), start + size)
            out.append({"doc": p["doc"], "page": p["page"], "text": ENCODER.decode(tokens[start:end])})
            if end == len(tokens):
                break
            start = end - overlap
    return out


chunks = chunk(pages)
chroma = chromadb.PersistentClient(path="chroma_db")
collection = chroma.get_or_create_collection("reports", metadata={"hnsw:space": "cosine"})

if collection.count() == 0:
    for start in range(0, len(chunks), 64):
        batch = chunks[start:start + 64]
        vectors = client.embeddings.create(
            model="text-embedding-3-small",
            input=[c["text"] for c in batch],
        )
        collection.add(
            ids=[f'{c["doc"]}-p{c["page"]}-c{start + i}' for i, c in enumerate(batch)],
            embeddings=[d.embedding for d in vectors.data],
            documents=[c["text"] for c in batch],
            metadatas=[{"doc": c["doc"], "page": c["page"]} for c in batch],
        )


def search_reports(query, k=5):
    """Return the k chunks closest to `query`, each with its document, page, and similarity."""
    vector = client.embeddings.create(model="text-embedding-3-small", input=[query]).data[0].embedding
    results = collection.query(
        query_embeddings=[vector],
        n_results=k,
        include=["documents", "metadatas", "distances"],
    )
    if not results["documents"] or not results["documents"][0]:
        return []
    return [
        {"text": text, "doc": meta["doc"], "page": meta["page"], "similarity": 1 - distance}
        for text, meta, distance in zip(
            results["documents"][0], results["metadatas"][0], results["distances"][0]
        )
    ]


print(f"{collection.count()} chunks in the collection")

## The two tools

`bearing_capacity` is the Terzaghi (1943) superposition for a strip footing,

    q_ult = c*Nc + gamma*D*Nq + 0.5*gamma*B*Ngamma

with the factors of Vesic (1973),

    Nq = exp(pi*tan(phi)) * tan(45 + phi/2)^2
    Nc = (Nq - 1) / tan(phi)
    Ngamma = 2*(Nq + 1)*tan(phi)

A tool the model can call needs a JSON schema: a name, a description telling the model when to
use it, and typed parameters. `strict` with `additionalProperties: false` and every field
required makes the model send arguments that match the schema exactly.

In [ ]:
import json
import math


def bearing_capacity(c_psf, phi_deg, gamma_pcf, B_ft, D_ft, FS=3.0):
    """Ultimate and allowable bearing capacity of a strip footing.

    Terzaghi (1943) superposition with the bearing capacity factors of Vesic (1973).
    Strip footing only: no shape, depth, inclination, or groundwater factors.
    """
    # TODO: compute Nq, Nc, and Ngamma from the equations in the markdown above, then
    # q_ult = c_psf*Nc + gamma_pcf*D_ft*Nq + 0.5*gamma_pcf*B_ft*Ngamma and q_allow = q_ult / FS.
    # Return a dict with Nc, Nq, Ngamma, q_ult_psf, q_ult_kPa, q_allow_psf, q_allow_kPa, and
    # formula. One psf is 0.047880 kPa.
    return {}


TOOLS = [
    {
        "type": "function",
        "name": "search_reports",
        "description": "Search the site investigation reports. Returns passages with document name and page number.",
        "parameters": {
            "type": "object",
            "properties": {
                "query": {"type": "string", "description": "What to look for, in plain English."},
                "k": {"type": "integer", "description": "How many passages to return."},
            },
            "required": ["query", "k"],
            "additionalProperties": False,
        },
        "strict": True,
    },
    {
        "type": "function",
        "name": "bearing_capacity",
        "description": "Terzaghi bearing capacity of a strip footing with Vesic factors. Returns q_ult and q_allow in psf and kPa with a factor of safety of 3.",
        "parameters": {
            "type": "object",
            "properties": {
                "c_psf": {"type": "number", "description": "Cohesion in pounds per square foot."},
                "phi_deg": {"type": "number", "description": "Friction angle in degrees."},
                "gamma_pcf": {"type": "number", "description": "Unit weight in pounds per cubic foot."},
                "B_ft": {"type": "number", "description": "Footing width in feet."},
                "D_ft": {"type": "number", "description": "Embedment depth below grade in feet."},
            },
            "required": ["c_psf", "phi_deg", "gamma_pcf", "B_ft", "D_ft"],
            "additionalProperties": False,
        },
        "strict": True,
    },
]


def call_tool(name, args):
    if name == "search_reports":
        return search_reports(**args)
    if name == "bearing_capacity":
        return bearing_capacity(**args)
    return {"error": f"unknown tool {name}"}

## The loop

One call to the model returns a list of output items. A `function_call` item carries a `name`, a
JSON string of `arguments`, and a `call_id`. The loop runs the named function and appends
`{"type": "function_call_output", "call_id": ..., "output": ...}` to the input list, then calls
again. The model's own items go back into the input too, so it sees what it already asked for.
The loop ends on the first step with no function calls.

`reasoning={"effort": "low", "summary": "auto"}` asks for a short summary of the model's internal
reasoning. Some accounts do not receive summaries, so the loop prints them only when present.

In [ ]:
REASONING = {"effort": "low", "summary": "auto"}


def run_agent(task, max_steps=10, client=client):
    instructions = (
        "You are a geotechnical engineer checking a foundation design against a site "
        "investigation report. Use search_reports to find every parameter before computing; "
        "cite the document and page for each. Use bearing_capacity for arithmetic; never "
        "compute by hand. If a parameter is missing from the report, state the assumption you "
        "make (for example unit weight 120 pcf). Finish with the computed allowable bearing "
        "pressure, the report's own recommendation with its page, and one paragraph on why "
        "they differ."
    )
    items = [{"role": "user", "content": task}]
    for step in range(max_steps):
        try:
            resp = client.responses.create(
                model=MODEL, instructions=instructions, input=items, tools=TOOLS,
                reasoning=REASONING,
            )
        except Exception as exc:
            # reasoning summaries need a verified organization; retry without them
            if "summary" not in str(exc).lower():
                raise
            REASONING.pop("summary", None)
            resp = client.responses.create(
                model=MODEL, instructions=instructions, input=items, tools=TOOLS,
                reasoning=REASONING,
            )
        items = items + list(resp.output)
        calls = []
        for item in resp.output:
            kind = getattr(item, "type", None)
            if kind == "reasoning":
                for part in getattr(item, "summary", None) or []:
                    text = getattr(part, "text", "")
                    if text:
                        print(f"[thinking] {text}")
            elif kind == "function_call":
                calls.append(item)
        if not calls:
            print(resp.output_text)
            return resp.output_text
        for item in calls:
            args = json.loads(item.arguments)
            print(f"-> {item.name}({args})")
            result = call_tool(item.name, args)
            if isinstance(result, list):
                line = "; ".join(f'{h["doc"]} p.{h["page"]}' for h in result)
            else:
                line = json.dumps(result)
            print(f"   {line[:200]}")
            # TODO: append the tool result so the model sees it on the next step. It is a dict
            # TODO: with type function_call_output, the item call_id, and json.dumps(result).
    print(f"Stopped after {max_steps} steps with no final answer.")
    return None

Now give the agent a design to check. Watch the trace: each `->` line is a tool the model chose
to call, with the arguments it chose.

In [ ]:
answer = run_agent("Using the Terracon report, estimate the allowable bearing pressure for a 4 ft wide strip footing embedded 3 ft below grade, and compare with the report's recommendation.")

## Reading the trace

The trace is the chain of work a reviewer can audit: what the agent looked up, which page it came
from, what it assumed where the report was silent, and what it computed. Compare that with 7a,
where the same kind of question produced a number with no source and no way to check it.

Expect the two bearing pressures to disagree by a wide margin. The Terracon recommendation of
2,000 psf in section 4.3.1 is set by settlement, one inch total and half an inch over 40 feet,
and by the near-surface soils, whose expansion index of 25 to 62 is why the footings bear on
engineered fill. A bearing failure check on the measured strength gives a number several times
larger. The agent should say so rather than call the difference an error.

## Check the tool by hand

The direct shear tests in the Terracon report give a friction angle of 29 to 30 degrees and a
cohesion of 432 to 539 psf. Taking the midpoints, `bearing_capacity(485, 29.5, 120, 4, 3)` gives
Nq 17.39, Nc 28.97, Ngamma 20.81, q_ult 25,305.7 psf (1,211.6 kPa), and q_allow 8,435.2 psf
(403.9 kPa). Compare these with the arguments and the result in the agent's tool call above.

In [ ]:
print(json.dumps(bearing_capacity(485, 29.5, 120, 4, 3), indent=2))